# Your L1 study: guided walkthrough
Run from the project folder after installing requirements. Synthetic results are only a code check. Read README first.


In [ ]:
from pathlib import Path
import pandas as pd
from pipeline import read_quotes, sample_quotes, make_features, split_time
from models import fit_models, evaluate


## 1. Read the bundled synthetic quotes
Replace the path with your verified market data only when ready.

In [ ]:
raw = read_quotes(Path('example_results/synthetic_quotes.csv'))
raw.head()

## 2. Inspect data quality and aligned targets
At timestamp t, future_mid must correspond to exactly t+1 second, never the next available distant quote.

In [ ]:
grid, audit = sample_quotes(raw, max_age=2)
f = make_features(grid, window=60)
print(audit)
f[['timestamp','quote_timestamp','mid','target_timestamp','future_mid','move','edge']].iloc[65:75]

## 3. Fit on earlier data, calibrate on the middle block
Scaling is fitted only on training. The test block is not used to fit either model.

In [ ]:
train, calibration, test = split_time(f)
models = fit_models(train, calibration)
print('OLS alpha:', models['edge'].intercept_, 'beta:', models['edge'].coef_[0])

## 4. Evaluate once
Tick 0.1 is for the synthetic example. Use actual source metadata for real data.

In [ ]:
predictions, regression, classification, probability, reports = evaluate(models, test, tick=.1)
display(regression, classification, probability)

## 5. View bundled graphs
Read INTERPRETATION.md. These examples contain deliberately planted synthetic signal.

In [ ]:
from IPython.display import Image, display
for path in sorted(Path('example_results/figures').glob('*.png')):
    display(Image(filename=str(path)))

## Your conclusions
Write observations on baseline improvement, weak regimes, probability quality and limitations. Use only real-data results as market evidence.